In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import pickle
import json

df = pd.read_csv("../data/Bengaluru_House_Data.csv")


In [2]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
df.tail()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.0
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.0
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.0
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.0
13319,Super built-up Area,Ready To Move,Doddathoguru,1 BHK,NaN,550,1.0,1.0,17.0


In [4]:
df.shape

(13320, 9)

In [5]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price'],
      dtype='str')

In [6]:
df.dtypes

area_type           str
availability        str
location            str
size                str
society             str
total_sqft          str
bath            float64
balcony         float64
price           float64
dtype: object

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  str    
 1   availability  13320 non-null  str    
 2   location      13319 non-null  str    
 3   size          13304 non-null  str    
 4   society       7818 non-null   str    
 5   total_sqft    13320 non-null  str    
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), str(6)
memory usage: 1.6 MB


In [8]:
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [9]:
df.nunique()

area_type          4
availability      81
location        1305
size              31
society         2688
total_sqft      2117
bath              19
balcony            4
price           1994
dtype: int64

In [10]:
df["area_type"].value_counts()

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64

In [11]:
df["availability"].value_counts()

availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
15-Aug               1
17-Jan               1
16-Nov               1
16-Jan               1
14-Jul               1
Name: count, Length: 81, dtype: int64

In [12]:
df["location"].value_counts().head(20)

location
Whitefield                  540
Sarjapur  Road              399
Electronic City             302
Kanakpura Road              273
Thanisandra                 234
Yelahanka                   213
Uttarahalli                 186
Hebbal                      177
Marathahalli                175
Raja Rajeshwari Nagar       171
Hennur Road                 152
Bannerghatta Road           152
7th Phase JP Nagar          149
Haralur Road                142
Electronic City Phase II    132
Rajaji Nagar                107
Chandapura                  100
Bellandur                    96
KR Puram                     91
Electronics City Phase 1     88
Name: count, dtype: int64

In [13]:
df["size"].value_counts()

size
2 BHK         5199
3 BHK         4310
4 Bedroom      826
4 BHK          591
3 Bedroom      547
1 BHK          538
2 Bedroom      329
5 Bedroom      297
6 Bedroom      191
1 Bedroom      105
8 Bedroom       84
7 Bedroom       83
5 BHK           59
9 Bedroom       46
6 BHK           30
7 BHK           17
1 RK            13
10 Bedroom      12
9 BHK            8
8 BHK            5
11 BHK           2
11 Bedroom       2
10 BHK           2
27 BHK           1
19 BHK           1
16 BHK           1
43 Bedroom       1
14 BHK           1
12 Bedroom       1
13 BHK           1
18 Bedroom       1
Name: count, dtype: int64

In [14]:
df["total_sqft"].head(20)

0     1056
1     2600
2     1440
3     1521
4     1200
5     1170
6     2732
7     3300
8     1310
9     1020
10    1800
11    2785
12    1000
13    1100
14    2250
15    1175
16    1180
17    1540
18    2770
19    1100
Name: total_sqft, dtype: str

In [15]:
df[~df["total_sqft"].str.replace(".", "", regex=False).str.replace("-", "", regex=False).str.strip().str.isnumeric()]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
30,Super built-up Area,19-Dec,Yelahanka,4 BHK,LedorSa,2100 - 2850,4.0,0.0,186.000
56,Built-up Area,20-Feb,Devanahalli,4 Bedroom,BrereAt,3010 - 3410,NaN,NaN,192.000
81,Built-up Area,18-Oct,Hennur Road,4 Bedroom,Gollela,2957 - 3450,NaN,NaN,224.500
122,Super built-up Area,18-Mar,Hebbal,4 BHK,SNontle,3067 - 8156,4.0,0.0,477.000
137,Super built-up Area,19-Mar,8th Phase JP Nagar,2 BHK,Vaarech,1042 - 1105,2.0,0.0,54.005
...,...,...,...,...,...,...,...,...,...
12990,Super built-up Area,18-May,Talaghattapura,3 BHK,Sodgere,1804 - 2273,3.0,0.0,122.000
13059,Super built-up Area,Ready To Move,Harlur,2 BHK,Shodsir,1200 - 1470,2.0,0.0,72.760
13240,Super built-up Area,Ready To Move,Devanahalli,1 BHK,Pardsri,1020 - 1130,NaN,NaN,52.570
13265,Super built-up Area,20-Sep,Hoodi,2 BHK,Ranuetz,1133 - 1384,2.0,0.0,59.135


In [16]:
df[~df["total_sqft"].str.replace(".", "", regex=False)
                   .str.replace("-", "", regex=False)
                   .str.strip()
                   .str.isnumeric()]["total_sqft"].unique()

<ArrowStringArray>
[    '2100 - 2850',     '3010 - 3410',     '2957 - 3450',     '3067 - 8156',
     '1042 - 1105',     '1145 - 1340',     '1015 - 1540',     '1520 - 1740',
  '34.46Sq. Meter',     '1195 - 1440',
 ...
     '1400 - 1421',     '4000 - 4450', '142.84Sq. Meter',    '300Sq. Yards',
     '2204 - 2362',     '1437 - 1629',      '850 - 1060',     '1200 - 1470',
     '1020 - 1130',     '1133 - 1384']
Length: 222, dtype: str

In [17]:
def convert_sqft_to_num(x):
    tokens = x.split("-")

    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2

    try:
        return float(x)
    except ValueError:
        return None

In [18]:
df["total_sqft"] = df["total_sqft"].apply(convert_sqft_to_num)

In [19]:
df["total_sqft"].dtype

dtype('float64')

In [20]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440.0,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200.0,2.0,1.0,51.00


In [21]:
df["total_sqft"].isnull().sum()

np.int64(46)

In [22]:
df[df["size"].isnull()]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
579,Plot Area,Immediate Possession,Sarjapur Road,NaN,Asiss B,1800.0,NaN,NaN,34.185
1775,Plot Area,Immediate Possession,IVC Road,NaN,Orana N,3817.0,NaN,NaN,124.000
2264,Plot Area,Immediate Possession,Banashankari,NaN,NaN,2400.0,NaN,NaN,460.000
2809,Plot Area,Immediate Possession,Sarjapur Road,NaN,AsdiaAr,1800.0,NaN,NaN,28.785
2862,Plot Area,Immediate Possession,Devanahalli,NaN,Ajleyor,1950.0,NaN,NaN,46.800
5333,Plot Area,Immediate Possession,Devanahalli,NaN,Emngs S,3752.5,NaN,NaN,177.115
6423,Plot Area,Immediate Possession,Whitefield,NaN,SRniaGa,2324.0,NaN,NaN,26.730
6636,Plot Area,Immediate Possession,Jigani,NaN,S2enste,1500.0,NaN,NaN,25.490
6719,Plot Area,Immediate Possession,Hoskote,NaN,SJowsn,1730.0,NaN,NaN,28.545
7680,Plot Area,Immediate Possession,Kasavanhalli,NaN,NaN,5000.0,NaN,NaN,400.000


In [23]:
df = df.dropna(subset=["size"])

In [24]:
df["size"].isnull().sum()

np.int64(0)

In [25]:
df["bhk"] = df["size"].str.split().str[0].astype(int)

In [26]:
df[["size", "bhk"]].head(10)

,size,bhk
0,2 BHK,2
1,4 Bedroom,4
2,3 BHK,3
3,3 BHK,3
4,2 BHK,2
5,2 BHK,2
6,4 BHK,4
7,4 BHK,4
8,3 BHK,3
9,6 Bedroom,6


In [27]:
df["sqft_per_bhk"] = df["total_sqft"] / df["bhk"]

In [28]:
df[["total_sqft", "bhk", "sqft_per_bhk"]].head(10)

,total_sqft,bhk,sqft_per_bhk
0,1056.0,2,528.000000
1,2600.0,4,650.000000
2,1440.0,3,480.000000
3,1521.0,3,507.000000
4,1200.0,2,600.000000
5,1170.0,2,585.000000
6,2732.0,4,683.000000
7,3300.0,4,825.000000
8,1310.0,3,436.666667
9,1020.0,6,170.000000


In [29]:
df[df["sqft_per_bhk"] < 300]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk,sqft_per_bhk
9,Plot Area,Ready To Move,Gandhi Bazar,6 Bedroom,NaN,1020.0,6.0,NaN,370.0,6,170.000000
45,Plot Area,Ready To Move,HSR Layout,8 Bedroom,NaN,600.0,9.0,NaN,200.0,8,75.000000
58,Plot Area,Ready To Move,Murugeshpalya,6 Bedroom,NaN,1407.0,4.0,1.0,150.0,6,234.500000
68,Plot Area,Ready To Move,Devarachikkanahalli,8 Bedroom,NaN,1350.0,7.0,0.0,85.0,8,168.750000
70,Plot Area,Ready To Move,Double Road,3 Bedroom,NaN,500.0,3.0,2.0,100.0,3,166.666667
...,...,...,...,...,...,...,...,...,...,...,...
13277,Plot Area,Ready To Move,Kundalahalli Colony,7 Bedroom,NaN,1400.0,7.0,NaN,218.0,7,200.000000
13279,Plot Area,Ready To Move,Vishwanatha Nagenahalli,6 Bedroom,NaN,1200.0,5.0,NaN,130.0,6,200.000000
13281,Plot Area,Ready To Move,Margondanahalli,5 Bedroom,NaN,1375.0,5.0,1.0,125.0,5,275.000000
13303,Plot Area,Ready To Move,Vidyaranyapura,5 Bedroom,NaN,774.0,5.0,3.0,70.0,5,154.800000


In [30]:
df[df["sqft_per_bhk"] < 300].shape

(744, 11)

In [31]:
df.shape

(13304, 11)

In [32]:
df1 = df[df["sqft_per_bhk"] >= 300]

In [33]:
df1.shape

(12514, 11)

In [34]:
len(df1["location"].unique())

1216

In [35]:
df1["location"] = df1["location"].str.strip()

In [36]:
location_stats = df1["location"].value_counts()

In [37]:
location_stats.head(20)

location
Whitefield                  537
Sarjapur  Road              393
Electronic City             295
Kanakpura Road              269
Thanisandra                 235
Yelahanka                   207
Uttarahalli                 180
Hebbal                      176
Marathahalli                170
Raja Rajeshwari Nagar       170
Hennur Road                 151
Bannerghatta Road           150
7th Phase JP Nagar          145
Haralur Road                141
Electronic City Phase II    127
Rajaji Nagar                101
Chandapura                   97
Bellandur                    95
Electronics City Phase 1     87
KR Puram                     83
Name: count, dtype: int64

In [38]:
len(location_stats[location_stats <= 10])

983

In [39]:
location_stats_less_than_10 = location_stats[location_stats <= 10]

In [40]:
df1["location"] = df1["location"].apply(
    lambda x: "other" if x in location_stats_less_than_10 else x
)

In [41]:
len(df1["location"].unique())

224

In [42]:
df1["price_per_sqft"] = (df1["price"] * 100000) / df1["total_sqft"]

In [43]:
df1[["price", "total_sqft", "price_per_sqft"]].head()

,price,total_sqft,price_per_sqft
0,39.07,1056.0,3699.810606
1,120.00,2600.0,4615.384615
2,62.00,1440.0,4305.555556
3,95.00,1521.0,6245.890861
4,51.00,1200.0,4250.000000


In [44]:
for key, subdf in df1.groupby("location"):
    print(key)
    print(subdf.head(2))
    break

1st Phase JP Nagar
                 area_type   availability            location   size  society  \
936   Super built-up  Area  Ready To Move  1st Phase JP Nagar  4 BHK  Prhtsok   
2106  Super built-up  Area  Ready To Move  1st Phase JP Nagar  3 BHK  Prhtsok   

      total_sqft  bath  balcony  price  bhk  sqft_per_bhk  price_per_sqft  
936       2825.0   4.0      3.0  250.0    4        706.25     8849.557522  
2106      1875.0   3.0      1.0  167.0    3        625.00     8906.666667  


In [45]:
type(subdf)

pandas.DataFrame

In [46]:
subdf.head()


,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk,sqft_per_bhk,price_per_sqft
936,Super built-up Area,Ready To Move,1st Phase JP Nagar,4 BHK,Prhtsok,2825.0,4.0,3.0,250.0,4,706.250000,8849.557522
2106,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,Prhtsok,1875.0,3.0,1.0,167.0,3,625.000000,8906.666667
2292,Built-up Area,Ready To Move,1st Phase JP Nagar,5 Bedroom,NaN,1500.0,5.0,NaN,85.0,5,300.000000,5666.666667
2914,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,Prhtsok,2065.0,4.0,1.0,210.0,3,688.333333,10169.491525
3481,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,NCaveun,2024.0,3.0,NaN,157.0,3,674.666667,7756.916996


In [47]:
subdf["price_per_sqft"].mean()

np.float64(9726.405917927166)

In [48]:
subdf.shape

(23, 12)

In [49]:
subdf["price_per_sqft"].std()

np.float64(4802.1275700779415)

In [50]:
def remove_pps_outliers(df):
    df_out = pd.DataFrame()

    for key, subdf in df.groupby("location"):
        m = np.mean(subdf["price_per_sqft"])
        st = np.std(subdf["price_per_sqft"])

        reduced_df = subdf[
            (subdf["price_per_sqft"] > (m - st)) &
            (subdf["price_per_sqft"] <= (m + st))
        ]

        df_out = pd.concat([df_out, reduced_df], ignore_index=True)

    return df_out

In [51]:
df2 = remove_pps_outliers(df1)

In [52]:
df2.shape

(10315, 12)

In [53]:
for location, location_df in df2.groupby("location"):
    print(location)
    print(location_df[["bhk", "price", "price_per_sqft"]].head())
    break

1st Phase JP Nagar
   bhk  price  price_per_sqft
0    4  250.0     8849.557522
1    3  167.0     8906.666667
2    5   85.0     5666.666667
3    3  210.0    10169.491525
4    3  157.0     7756.916996


In [54]:
def remove_bhk_outliers(df):
    exclude_indices = np.array([])

    for location, location_df in df.groupby("location"):
        bhk_stats = {}

        for bhk, bhk_df in location_df.groupby("bhk"):
            bhk_stats[bhk] = {
                "mean": np.mean(bhk_df["price_per_sqft"]),
                "std": np.std(bhk_df["price_per_sqft"]),
                "count": bhk_df.shape[0]
            }

        for bhk, bhk_df in location_df.groupby("bhk"):
            stats = bhk_stats.get(bhk - 1)

            if stats and stats["count"] > 5:
                exclude_indices = np.append(
                    exclude_indices,
                    bhk_df[bhk_df["price_per_sqft"] < stats["mean"]].index.values
                )

    return df.drop(exclude_indices, axis="index")

In [55]:
df3 = remove_bhk_outliers(df2)

In [56]:
df3.shape

(7289, 12)

In [57]:
df4 = df3.drop(["size", "price_per_sqft"], axis=1)

In [58]:
df4.head()

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,sqft_per_bhk
1,Super built-up Area,Ready To Move,1st Phase JP Nagar,Prhtsok,1875.0,3.0,1.0,167.0,3,625.000000
2,Built-up Area,Ready To Move,1st Phase JP Nagar,NaN,1500.0,5.0,NaN,85.0,5,300.000000
3,Super built-up Area,Ready To Move,1st Phase JP Nagar,Prhtsok,2065.0,4.0,1.0,210.0,3,688.333333
5,Super built-up Area,Ready To Move,1st Phase JP Nagar,Prhtsok,2059.0,3.0,2.0,225.0,3,686.333333
6,Super built-up Area,Ready To Move,1st Phase JP Nagar,NCaveun,1394.0,2.0,1.0,100.0,2,697.000000


In [59]:
df4 = df4.reset_index(drop=True)

In [60]:
df4.head()

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,sqft_per_bhk
0,Super built-up Area,Ready To Move,1st Phase JP Nagar,Prhtsok,1875.0,3.0,1.0,167.0,3,625.000000
1,Built-up Area,Ready To Move,1st Phase JP Nagar,NaN,1500.0,5.0,NaN,85.0,5,300.000000
2,Super built-up Area,Ready To Move,1st Phase JP Nagar,Prhtsok,2065.0,4.0,1.0,210.0,3,688.333333
3,Super built-up Area,Ready To Move,1st Phase JP Nagar,Prhtsok,2059.0,3.0,2.0,225.0,3,686.333333
4,Super built-up Area,Ready To Move,1st Phase JP Nagar,NCaveun,1394.0,2.0,1.0,100.0,2,697.000000


In [61]:
df4.dtypes

area_type           str
availability        str
location            str
society             str
total_sqft      float64
bath            float64
balcony         float64
price           float64
bhk               int64
sqft_per_bhk    float64
dtype: object

In [62]:
df5 = df4.drop(["availability", "society"], axis=1)

In [63]:
df5.head()

,area_type,location,total_sqft,bath,balcony,price,bhk,sqft_per_bhk
0,Super built-up Area,1st Phase JP Nagar,1875.0,3.0,1.0,167.0,3,625.000000
1,Built-up Area,1st Phase JP Nagar,1500.0,5.0,NaN,85.0,5,300.000000
2,Super built-up Area,1st Phase JP Nagar,2065.0,4.0,1.0,210.0,3,688.333333
3,Super built-up Area,1st Phase JP Nagar,2059.0,3.0,2.0,225.0,3,686.333333
4,Super built-up Area,1st Phase JP Nagar,1394.0,2.0,1.0,100.0,2,697.000000


In [64]:
dummies = pd.get_dummies(df5[["area_type", "location"]], dtype=int)

In [65]:
dummies.head()

,area_type_Built-up Area,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area,location_1st Phase JP Nagar,location_2nd Phase Judicial Layout,location_5th Phase JP Nagar,location_6th Phase JP Nagar,location_7th Phase JP Nagar,location_8th Phase JP Nagar,...,location_Vidyaranyapura,location_Vijayanagar,location_Vittasandra,location_Whitefield,location_Yelachenahalli,location_Yelahanka,location_Yelahanka New Town,location_Yelenahalli,location_Yeshwanthpur,location_other
0,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [66]:
dummies = dummies.drop(["area_type_Built-up  Area", "location_other"], axis=1)

In [67]:
dummies.columns

Index(['area_type_Carpet  Area', 'area_type_Plot  Area',
       'area_type_Super built-up  Area', 'location_1st Phase JP Nagar',
       'location_2nd Phase Judicial Layout', 'location_5th Phase JP Nagar',
       'location_6th Phase JP Nagar', 'location_7th Phase JP Nagar',
       'location_8th Phase JP Nagar', 'location_9th Phase JP Nagar',
       ...
       'location_Vasanthapura', 'location_Vidyaranyapura',
       'location_Vijayanagar', 'location_Vittasandra', 'location_Whitefield',
       'location_Yelachenahalli', 'location_Yelahanka',
       'location_Yelahanka New Town', 'location_Yelenahalli',
       'location_Yeshwanthpur'],
      dtype='str', length=225)

In [68]:
df6 = df5.drop(["area_type", "location"], axis=1)

In [69]:
df7 = pd.concat([df6, dummies], axis=1)

In [70]:
df7.head()

,total_sqft,bath,balcony,price,bhk,sqft_per_bhk,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area,location_1st Phase JP Nagar,...,location_Vasanthapura,location_Vidyaranyapura,location_Vijayanagar,location_Vittasandra,location_Whitefield,location_Yelachenahalli,location_Yelahanka,location_Yelahanka New Town,location_Yelenahalli,location_Yeshwanthpur
0,1875.0,3.0,1.0,167.0,3,625.000000,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1500.0,5.0,NaN,85.0,5,300.000000,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,2065.0,4.0,1.0,210.0,3,688.333333,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,2059.0,3.0,2.0,225.0,3,686.333333,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,1394.0,2.0,1.0,100.0,2,697.000000,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [71]:
df7.shape

(7289, 231)

In [72]:
X = df7.drop("price", axis=1)

In [73]:
y = df7["price"]

In [74]:
X.shape

(7289, 230)

In [75]:
y.shape

(7289,)

In [76]:
from sklearn.model_selection import train_test_split

In [77]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=10
)

In [78]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5831, 230)
(1458, 230)
(5831,)
(1458,)


In [79]:
model = LinearRegression()

In [80]:
df7.isnull().sum()

total_sqft                       0
bath                            32
balcony                        256
price                            0
bhk                              0
                              ... 
location_Yelachenahalli          0
location_Yelahanka               0
location_Yelahanka New Town      0
location_Yelenahalli             0
location_Yeshwanthpur            0
Length: 231, dtype: int64

In [81]:
df7[df7.isnull().any(axis=1)].shape

(256, 231)

In [82]:
df7 = df7.dropna()

In [83]:
X = df7.drop("price", axis=1)
y = df7["price"]

In [84]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=10
)

In [85]:
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](230,)","[ 0.12, -1.62, -1.01,...,-26.52,-50.97,-10.07]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](230,)","['total_sqft','bath','balcony',...,'location_Yelahanka New Town', 'location_Yelenahalli','location_Yeshwanthpur']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,53.59
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,230
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(230)


In [86]:
df7.isnull().sum().sum()

np.int64(0)

In [87]:
y_pred = model.predict(X_test)

In [88]:
score = r2_score(y_test, y_pred)
print(score)

0.855661129363288


In [89]:
cv = ShuffleSplit(
    n_splits=5,
    test_size=0.2,
    random_state=0
)

In [90]:
scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=cv
)

In [91]:
print(scores)
print(scores.mean())

[0.86679694 0.83802723 0.84362004 0.85114196 0.88975695]
0.8578686234816638


In [92]:
final_model = LinearRegression()
final_model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](230,)","[ 0.12, 0.61, -1.04,...,-26.85,-51.17,-10.62]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](230,)","['total_sqft','bath','balcony',...,'location_Yelahanka New Town', 'location_Yelenahalli','location_Yeshwanthpur']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,50.29
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,230
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(230)


In [94]:
import pickle

with open("../models/bangalore_house_price_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

In [95]:
import json

columns = {
    "data_columns": list(X.columns)
}

with open("../models/columns.json", "w") as f:
    json.dump(columns, f, indent=4)

In [96]:
print(X.columns.tolist()[:10])
print(X.columns.tolist()[-10:])

['total_sqft', 'bath', 'balcony', 'bhk', 'sqft_per_bhk', 'area_type_Carpet  Area', 'area_type_Plot  Area', 'area_type_Super built-up  Area', 'location_1st Phase JP Nagar', 'location_2nd Phase Judicial Layout']
['location_Vasanthapura', 'location_Vidyaranyapura', 'location_Vijayanagar', 'location_Vittasandra', 'location_Whitefield', 'location_Yelachenahalli', 'location_Yelahanka', 'location_Yelahanka New Town', 'location_Yelenahalli', 'location_Yeshwanthpur']
